# Train the IO↔EO cross-encoder re-ranker

Trains an `xlm-roberta-base` + binary classification head on Wiktionary +
Wikipedia-langlink positives, BERT top-K hard negatives, and edit-distance
surface negatives. Outputs `models/cross-encoder-io-eo/` (~500 MB) for
consumption by stage 19 (re-ranking inference, PR B).

**Runtime**: T4 GPU, ~30–45 min wall-clock end-to-end.

**Acceptance**: F1 ≥ 0.85 on the held-out 200-pair eval set.

## 1. Verify GPU

Should show Tesla T4 with ~15 GB VRAM. If not: Runtime → Change runtime type → T4 GPU.

In [ ]:
!nvidia-smi

## 2. Clone the repo + install deps

In [ ]:
!git clone --depth=1 https://github.com/komapc/embedding-aligner.git
%cd embedding-aligner
!pip install -q transformers==4.46.3 datasets accelerate scikit-learn

## 3. Fetch extractor inputs

`bilingual_raw.json` and `io_eo_langlinks.json` are too big for git; they
live as a release asset. ~1.8 MB compressed → ~48 MB uncompressed.

In [ ]:
!wget -q https://github.com/komapc/embedding-aligner/releases/download/cross-encoder-inputs/cross_encoder_inputs.tar.gz
!mkdir -p extractor_work
!tar xzf cross_encoder_inputs.tar.gz -C extractor_work --strip-components=1
!ls -lh extractor_work/

## 4. Train (3 epochs, ~25 min on T4)

Watch for:
1. `Wiktionary positives: ~38k` and `Langlink positives: ~3.7k`
2. `Split: train=~39k heldout=200`
3. `Hard negatives: ~22k`, `Surface-similar negatives: ~30–40k` (slow CPU
   step, ~10 min — `difflib` doesn't use GPU)
4. Training: ~3 epochs × ~3000 steps with FP16
5. `Final held-out metrics: {'accuracy': ..., 'f1': ..., 'auc': ...}`

In [ ]:
!python3 scripts/16_train_cross_encoder.py \
  --bilingual-raw extractor_work/bilingual_raw.json \
  --langlinks extractor_work/io_eo_langlinks.json \
  --candidates results/bert_ido_epo_alignment/translation_candidates.json \
  --eo-vocab data/esperanto_vocabulary.txt \
  --heldout-out data/cross_encoder_heldout.jsonl \
  --model-out models/cross-encoder-io-eo \
  --epochs 3 \
  --batch-size 32

## 5. Download trained model

~500 MB — drops to your `~/Downloads/`. Extract on your laptop with:
```
mkdir -p ~/projects/apertium-dev/projects/embedding-aligner/models
tar xzf ~/Downloads/cross-encoder-io-eo.tar.gz \
  -C ~/projects/apertium-dev/projects/embedding-aligner/
```

In [ ]:
!tar czf cross-encoder-io-eo.tar.gz models/cross-encoder-io-eo
from google.colab import files
files.download('cross-encoder-io-eo.tar.gz')